# 🚀 LLM 初心者向けハンズオン実践
**Python 開発経験者向け - 実装を通じて学ぶ**

このノートブックでは、以下を実装します：
1. 最初の推論（Hello LLM）
2. Tokenizer の詳細動作
3. 埋め込みベクトルの可視化
4. 簡易 RAG システム
5. 自律性スコアリング
6. インタラクティブな演習問題

## 📦 セットアップ

必要なライブラリをインストール・インポート

In [1]:
# 必要なライブラリのインストール（初回のみ）
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package)
        print(f"✓ {package} はインストール済みです")
    except ImportError:
        print(f"インストール中: {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} をインストールしました")

# 必要なパッケージ
packages = ['torch', 'transformers', 'datasets', 'numpy', 'pandas', 'scikit-learn', 'matplotlib']

for package in packages:
    install_if_missing(package)

print("\n✅ すべての準備が完了しました！")

✓ torch はインストール済みです


/home/abemc/project_root/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ transformers はインストール済みです
✓ datasets はインストール済みです
✓ numpy はインストール済みです
✓ pandas はインストール済みです
インストール中: scikit-learn...
✓ scikit-learn をインストールしました
✓ matplotlib はインストール済みです

✅ すべての準備が完了しました！


In [2]:
# ライブラリのインポート
import torch
import numpy as np
import pandas as pd
from typing import Any
from transformers import AutoTokenizer, AutoModel, pipeline  # type: ignore
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（文字化け対策）
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 設定
plt.style.use('seaborn-v0_8-darkgrid')
torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)

print(f"PyTorch バージョン: {torch.__version__}")
print(f"Transformers バージョン: {AutoTokenizer.__module__}")
print(f"GPU 利用可能: {torch.cuda.is_available()}")
print(f"使用デバイス: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

PyTorch バージョン: 2.11.0+cu130
Transformers バージョン: transformers.models.auto.tokenization_auto
GPU 利用可能: True
使用デバイス: cuda


## 1️⃣ 最初の推論：Hello LLM

Hugging Face のパイプラインを使った最も簡単な推論

In [4]:
# type: ignore
# 例1: テキスト分類（感情分析）
print("="*60)
print("例1: 感情分析")
print("="*60)

# ✅ ステップ1: 分類パイプラインの作成
# 感情分析用の事前学習済みモデルをロード
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# ✅ ステップ2: テスト用のテキストを準備
texts = [
    "I absolutely love this product!",
    "This is terrible and disappointing.",
    "It's okay, nothing special."
]

# ✅ ステップ3: テキスト全体を分類器に渡す（バッチ処理）
results = classifier(texts)

# ✅ ステップ4: 結果を整形して表示
for text, result in zip(texts, results):                                    # zip 関数を使用して、テキストと結果を同時にループ処理
    print(f"\nテキスト: {text}")                                            # test キーを使用して、入力テキストを表示
    print(f"  感情: {result['label']}")                                     # label キーを使用して、感情のラベルを表示  result キーを使用して、確信度を表示
    print(f"  確信度: {result['score']:.4f} ({result['score']*100:.2f}%)")  #

例1: 感情分析


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 299.56it/s]



テキスト: I absolutely love this product!
  感情: POSITIVE
  確信度: 0.9999 (99.99%)

テキスト: This is terrible and disappointing.
  感情: NEGATIVE
  確信度: 0.9998 (99.98%)

テキスト: It's okay, nothing special.
  感情: NEGATIVE
  確信度: 0.8190 (81.90%)


### 📊 感情分析の仕組みを詳しく理解

感情（label）と確信度（score）がどのように判定されているのか、内部動作を見てみましょう


In [10]:
# ✅ ステップ1: 感情分析モデルの仕組み

# セル間の依存関係を解消: tokenizer を定義
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# デバイスの設定（GPU/CPU の互換性確保）
device = classifier.model.device

print("="*60)
print("感情分析モデルの内部動作")
print("="*60)

print("""
使用モデル: distilbert-base-uncased-finetuned-sst-2-english

モデルの構成:
1. DistilBERT ベースモデル
   - 事前学習済みの汎用言語モデル
   - テキストの意味を理解する基盤

2. ファインチューニング (SST-2 データセット)
   - 感情分類タスクに特化した学習
   - 肯定的 (POSITIVE) / 否定的 (NEGATIVE) の2分類

3. 出力層
   - ロジット（生スコア）を確率に変換
   - Softmax 関数により合計が1になる確率に
""")

# ✅ ステップ2: 1つのテキストの詳細分析
print("\n" + "="*60)
print("テキスト1つの内部処理を詳しく見る")
print("="*60)

test_text = "I absolutely love this product!"
print(f"\n📝 入力テキスト: {test_text}\n")

# トークン化
print("ステップ1: トークン化")
tokens = tokenizer.tokenize(test_text)
print(f"  トークン: {tokens}")
print(f"  トークン数: {len(tokens)}\n")

# エンコーディング
print("ステップ2: 数値エンコーディング")
encoded = tokenizer(test_text, return_tensors='pt', truncation=True, max_length=512)
# デバイスに移動（GPU/CPU 互換性確保）
encoded = {key: value.to(device) for key, value in encoded.items()}
print(f"  Input IDs 形状: {encoded['input_ids'].shape}")
print(f"  Attention Mask 形状: {encoded['attention_mask'].shape}")
print(f"  デバイス: {encoded['input_ids'].device}\n")

# モデルの推論（内部ロジット取得）
print("ステップ3: モデルの推論（ロジット取得）")
with torch.no_grad():
    outputs = classifier.model(**encoded)
    logits = outputs.logits
    
print(f"  ロジット形状: {logits.shape}")
print(f"  ロジット値: {logits.cpu().numpy()}")
print(f"    → NEGATIVE (否定) のロジット: {logits[0][0].item():.4f}")
print(f"    → POSITIVE (肯定) のロジット: {logits[0][1].item():.4f}\n")

# Softmax で確率に変換
print("ステップ4: Softmax で確率に変換")
probabilities = torch.softmax(logits, dim=-1)
print(f"  確率: {probabilities.cpu().numpy()}")
print(f"    → NEGATIVE (否定) の確率: {probabilities[0][0].item():.4f}")
print(f"    → POSITIVE (肯定) の確率: {probabilities[0][1].item():.4f}\n")

# 最大値（最も確信度の高い感情）を取得
print("ステップ5: 最大確率のラベルと確信度")
class_id = probabilities.argmax().item()
score = probabilities.max().item()
label = classifier.model.config.id2label[class_id]

print(f"  最大確率のクラスID: {class_id}")
print(f"  対応するラベル: {label}")
print(f"  確信度 (スコア): {score:.4f} ({score*100:.2f}%)\n")

# ✅ ステップ3: 複数テキストの比較
print("="*60)
print("複数テキストの比較分析")
print("="*60)

comparison_texts = [
    "I absolutely love this product!",           # 強い肯定
    "This is okay.",                             # 弱い肯定
    "I like it.",                                # 中程度の肯定
    "This is terrible.",                         # 強い否定
]

print("\n比較結果:")
print("-" * 60)
print(f"{'テキスト':<35} | {'感情':<10} | {'確信度':<8}")
print("-" * 60)

for text in comparison_texts:
    result = classifier(text)[0]
    label = result['label']
    score = result['score']
    
    # テキスト表示を短縮
    display_text = text if len(text) <= 32 else text[:29] + "..."
    
    print(f"{display_text:<35} | {label:<10} | {score:>6.2%}")

print("\n📌 ポイント:")
print("  - 肯定的な単語が多い → スコアが高い")
print("  - 否定的な単語が多い → スコアが低い（NEGATIVE）")
print("  - 感情が曖昧 → スコアが 50% に近づく")


感情分析モデルの内部動作

使用モデル: distilbert-base-uncased-finetuned-sst-2-english

モデルの構成:
1. DistilBERT ベースモデル
   - 事前学習済みの汎用言語モデル
   - テキストの意味を理解する基盤

2. ファインチューニング (SST-2 データセット)
   - 感情分類タスクに特化した学習
   - 肯定的 (POSITIVE) / 否定的 (NEGATIVE) の2分類

3. 出力層
   - ロジット（生スコア）を確率に変換
   - Softmax 関数により合計が1になる確率に


テキスト1つの内部処理を詳しく見る

📝 入力テキスト: I absolutely love this product!

ステップ1: トークン化
  トークン: ['i', 'absolutely', 'love', 'this', 'product', '!']
  トークン数: 6

ステップ2: 数値エンコーディング
  Input IDs 形状: torch.Size([1, 8])
  Attention Mask 形状: torch.Size([1, 8])
  デバイス: cuda:0

ステップ3: モデルの推論（ロジット取得）
  ロジット形状: torch.Size([1, 2])
  ロジット値: [[-4.3579  4.7166]]
    → NEGATIVE (否定) のロジット: -4.3579
    → POSITIVE (肯定) のロジット: 4.7166

ステップ4: Softmax で確率に変換
  確率: [[0.0001 0.9999]]
    → NEGATIVE (否定) の確率: 0.0001
    → POSITIVE (肯定) の確率: 0.9999

ステップ5: 最大確率のラベルと確信度
  最大確率のクラスID: 1
  対応するラベル: POSITIVE
  確信度 (スコア): 0.9999 (99.99%)

複数テキストの比較分析

比較結果:
------------------------------------------------------------
テキスト                     

In [ ]:
# ✅ ステップ1: アーキテクチャと計算フロー

print("="*60)
print("感情分析の完全フロー")
print("="*60)

print("""
📊 処理の流れ:

1️⃣ 入力テキスト
   "I absolutely love this product!"

2️⃣ トークナイゼーション
   ["i", "absolute", "##ly", "love", "this", "product", "!"]
   → BEIRフォーマットのトークンに分割

3️⃣ トークン ID 変換
   [1, 4769, 2135, 2572, 2054, 4031, 999]
   → 語彙表のインデックスに変換

4️⃣ BERT エンコーディング
   特殊トークンを追加: [CLS] + tokens + [SEP]
   → 768 次元の隠れ状態を生成
   
5️⃣ [CLS] トークンの埋め込み
   クラス全体を代表するベクトル
   形状: (1, 768)

6️⃣ 感情分類層
   Dense(768 → 2)
   ロジット: [x_negative, x_positive]

7️⃣ Softmax 関数
   確率 = exp(x) / sum(exp(x))
   NEGATIVE: 0.01 (1%)
   POSITIVE: 0.99 (99%)

8️⃣ 出力
   ラベル: POSITIVE
   スコア: 0.99
""")

# ✅ ステップ2: 数学的な詳細

print("\n" + "="*60)
print("Softmax 関数の計算例")
print("="*60)

import numpy as np

# 仮想的なロジット値
logits_example = np.array([[-2.5, 3.2]])  # [negative, positive]

print(f"\n入力ロジット:")
print(f"  Negative: {logits_example[0][0]:.1f}")
print(f"  Positive: {logits_example[0][1]:.1f}")

# 手動で Softmax を計算
exp_logits = np.exp(logits_example[0])
softmax_probs = exp_logits / np.sum(exp_logits)

print(f"\nSoftmax 計算:")
print(f"  exp(Negative) = exp({logits_example[0][0]:.1f}) = {exp_logits[0]:.6f}")
print(f"  exp(Positive) = exp({logits_example[0][1]:.1f}) = {exp_logits[1]:.6f}")
print(f"  合計 = {np.sum(exp_logits):.6f}")
print(f"\n確率:")
print(f"  P(Negative) = {exp_logits[0]:.6f} / {np.sum(exp_logits):.6f} = {softmax_probs[0]:.4f} ({softmax_probs[0]*100:.2f}%)")
print(f"  P(Positive) = {exp_logits[1]:.6f} / {np.sum(exp_logits):.6f} = {softmax_probs[1]:.4f} ({softmax_probs[1]*100:.2f}%)")

# ✅ ステップ3: モデル情報

print("\n" + "="*60)
print("モデルの詳細情報")
print("="*60)

print(f"""
モデル名: distilbert-base-uncased-finetuned-sst-2-english

🔍 モデルの特性:
  - アーキテクチャ: DistilBERT (BERT を蒸留)
  - パラメータ数: 67M (BERTの40%削減)
  - 処理速度: BERTの60倍高速
  - 精度: BERT と同等レベル

📚 訓練データ:
  - SST-2 (Stanford Sentiment Treebank)
  - 映画レビュー67K + テスト1.8K
  - ラベル: POSITIVE / NEGATIVE

🎯 タスク:
  - 二値分類（Binary Classification）
  - テキストが肯定的か否定的かの判定

⚙️ 言語:
  - 英語専用
  - uncased: 大文字小文字を区別しない
""")

print(f"\nモデルの設定:")
print(f"  隠れ層サイズ: {classifier.model.config.hidden_size}")
print(f"  注意ヘッド数: {classifier.model.config.num_attention_heads}")
print(f"  層数: {classifier.model.config.num_hidden_layers}")
print(f"  ラベル: {classifier.model.config.id2label}")


In [ ]:
# ✅ ステップ1: 感情スコア分析の実践

print("="*60)
print("感情スコアの実践的活用")
print("="*60)

# ✅ ステップ2: 複数のレビューを分析

reviews = [
    "Perfect! Exactly what I needed. Highly recommend!",
    "Good quality, fast delivery, very satisfied.",
    "It's okay. Does the job but nothing special.",
    "Disappointed. Poor quality and broke after a week.",
    "Absolutely terrible! Total waste of money.",
]

print("\n📊 レビュー分析結果:")
print("="*80)

results_detailed = []
for i, review in enumerate(reviews, 1):
    result = classifier(review)[0]
    label = result['label']
    score = result['score']
    
    # スコアの解釈
    if label == 'POSITIVE':
        confidence = "高い肯定" if score > 0.9 else "中程度の肯定" if score > 0.7 else "弱い肯定"
    else:
        confidence = "高い否定" if score > 0.9 else "中程度の否定" if score > 0.7 else "弱い否定"
    
    results_detailed.append({
        'review': review,
        'label': label,
        'score': score,
        'confidence': confidence
    })
    
    # 視覚的な表示
    bar_length = int(score * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    
    print(f"\n{i}. {review}")
    print(f"   感情: {label} | 確信度: {score:.2%}")
    print(f"   [{bar}]")
    print(f"   判定: {confidence}")

# ✅ ステップ3: 統計分析

print("\n" + "="*60)
print("統計サマリー")
print("="*60)

df_results = pd.DataFrame(results_detailed)

positive_count = (df_results['label'] == 'POSITIVE').sum()
negative_count = (df_results['label'] == 'NEGATIVE').sum()
avg_positive_score = df_results[df_results['label'] == 'POSITIVE']['score'].mean()
avg_negative_score = df_results[df_results['label'] == 'NEGATIVE']['score'].mean()

print(f"\n総レビュー数: {len(reviews)}")
print(f"肯定的: {positive_count} ({positive_count/len(reviews)*100:.1f}%)")
print(f"否定的: {negative_count} ({negative_count/len(reviews)*100:.1f}%)")
print(f"\n平均確信度:")
print(f"  肯定的レビュー: {avg_positive_score:.2%}")
print(f"  否定的レビュー: {avg_negative_score:.2%}")

# ✅ ステップ4: 閾値の重要性

print("\n" + "="*60)
print("確信度の閾値による影響")
print("="*60)

print("""
実務での活用例:

❌ 低い確信度（50-70%）
   - 感情が曖昧
   - 人間のレビューが必要
   - 自動判定避けるべき

✅ 高い確信度（85%以上）
   - 確実な判定
   - 自動処理可能
   - ビジネスロジックに使用

⚠️  中程度（70-85%）
   - 注視が必要
   - 追加検証推奨
   - 文脈確認

例:
  - スコア 0.95 → 確実に肯定的
  - スコア 0.52 → 判定が曖昧（信頼度低い）
  - スコア 0.15 → 確実に否定的
""")

# ✅ ステップ5: 限界と改善

print("\n" + "="*60)
print("モデルの限界と改善")
print("="*60)

limitations = {
    "複雑な感情": "皮肉や冗談を理解しにくい",
    "コンテキスト不足": "複数センテンス間の関係を見落とす",
    "専門用語": "業界特有の用語に弱い可能性",
    "言語対応": "英語専用（日本語は不可）",
    "極端な表現": "極端な表現に過剰反応する可能性"
}

for issue, description in limitations.items():
    print(f"\n❌ {issue}")
    print(f"   → {description}")

print("\n\n✨ 改善方法:")
improvements = [
    "1. 複数モデルのアンサンブル",
    "2. ドメイン固有のファインチューニング",
    "3. 複数言語対応モデルの使用",
    "4. 感情スコアの信頼区間推定",
    "5. エラー分析と継続的改善"
]

for improvement in improvements:
    print(f"  {improvement}")


In [ ]:
# ✅ ステップ1: 感情スコアの可視化

print("="*60)
print("感情スコアの可視化")
print("="*60)

# テスト用の異なる感情レベルのテキスト
test_samples = [
    ("I absolutely love this!", 1),
    ("Really good product", 2),
    ("It's fine, nothing special", 3),
    ("Not great", 4),
    ("Terrible, waste of money", 5),
]

# 分析実行
sentiment_data = []
for text, idx in test_samples:
    result = classifier(text)[0]
    sentiment_data.append({
        'text': text,
        'idx': idx,
        'positive_score': result['score'] if result['label'] == 'POSITIVE' else 1 - result['score'],
        'label': result['label'],
        'raw_score': result['score']
    })

df_sentiment = pd.DataFrame(sentiment_data)

# ✅ ステップ2: グラフ1 - 水平棒グラフ

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# グラフ1: 肯定的スコアの水平棒グラフ
colors = ['green' if x > 0.5 else 'red' for x in df_sentiment['positive_score']]
axes[0].barh(range(len(df_sentiment)), df_sentiment['positive_score'], color=colors, alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(len(df_sentiment)))
axes[0].set_yticklabels([f"{row['text'][:25]}" for _, row in df_sentiment.iterrows()], fontsize=10)
axes[0].set_xlabel('Positive Score (肯定スコア)', fontsize=11)
axes[0].set_title('テキスト別の感情スコア', fontsize=12, fontweight='bold')
axes[0].set_xlim([0, 1])
axes[0].axvline(x=0.5, color='gray', linestyle='--', linewidth=2, label='中立線')
axes[0].grid(axis='x', alpha=0.3)
axes[0].legend()

# グラフ2: 円グラフ（全体分布）
label_counts = df_sentiment['label'].value_counts()
colors_pie = ['green', 'red']
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors_pie[:len(label_counts)], startangle=90, 
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('感情ラベルの分布', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# ✅ ステップ3: 詳細なスコア表示

print("\n詳細なスコア分析:")
print("="*80)
print(f"{'テキスト':<30} | {'感情':<10} | {'スコア':<8} | {'確信度':<8}")
print("="*80)

for _, row in df_sentiment.iterrows():
    text_display = row['text'][:27] if len(row['text']) > 27 else row['text']
    confidence = "高" if row['raw_score'] > 0.9 else "中" if row['raw_score'] > 0.7 else "低"
    print(f"{text_display:<30} | {row['label']:<10} | {row['raw_score']:>6.2%} | {confidence:<8}")

# ✅ ステップ4: スコア分布の統計

print("\n" + "="*60)
print("スコア統計")
print("="*60)

print(f"\n肯定的判定:")
positive_df = df_sentiment[df_sentiment['label'] == 'POSITIVE']
if len(positive_df) > 0:
    print(f"  平均スコア: {positive_df['raw_score'].mean():.2%}")
    print(f"  最大スコア: {positive_df['raw_score'].max():.2%}")
    print(f"  最小スコア: {positive_df['raw_score'].min():.2%}")

print(f"\n否定的判定:")
negative_df = df_sentiment[df_sentiment['label'] == 'NEGATIVE']
if len(negative_df) > 0:
    print(f"  平均スコア: {negative_df['raw_score'].mean():.2%}")
    print(f"  最大スコア: {negative_df['raw_score'].max():.2%}")
    print(f"  最小スコア: {negative_df['raw_score'].min():.2%}")

print("\n📌 重要なポイント:")
print("""
1. スコア 0.5 より高い → 肯定的（POSITIVE）
2. スコア 0.5 より低い → 否定的（NEGATIVE）
3. 確信度が高い（0.9以上）→ 判定に信頼がおける
4. 確信度が低い（0.5-0.6付近）→ 判定が曖昧
""")


In [ ]:
# type: ignore
# 例2: ゼロショット分類
print("\n" + "="*60)
print("例2: ゼロショット分類（テキスト分類の高度な例）")
print("="*60)

# ✅ ステップ1: ゼロショット分類パイプラインを作成
# 訓練データなしで複数ラベルの分類ができる
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)


例2: ゼロショット分類（テキスト分類の高度な例）


Loading weights: 100%|██████████| 515/515 [00:02<00:00, 185.29it/s]



テキスト: The weather is beautiful today. I love sunny days!
候補ラベル: ['weather', 'emotion', 'time', 'activity']

分類結果:
  weather         スコア: 0.8018 (80.18%)
  emotion         スコア: 0.0716 (7.16%)
  time            スコア: 0.0709 (7.09%)
  activity        スコア: 0.0557 (5.57%)

最も関連性の高いラベル: weather (0.8018)


## 2️⃣ Tokenizer の詳細動作

テキストがどのようにトークン化されるのか詳しく見てみましょう

In [ ]:
# ✅ ステップ1: 事前学習済みTokenizerをロード
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print("="*60)
print("Tokenizer の動作を追跡")
print("="*60)

# ✅ ステップ2: テスト用テキストを準備
text = "Hello, how are you? I'm learning about LLMs!"

print(f"\n元のテキスト: {text}")
print("-" * 60)

# ✅ ステップ3a: テキスト → トークンに分割
tokens = tokenizer.tokenize(text)
print(f"\nステップ3a: トークン化（テキスト → トークン）")
print(f"トークン: {tokens}")
print(f"トークン数: {len(tokens)}")

# ✅ ステップ3b: トークン → ID に変換
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"\nステップ3b: トークンID化（トークン → ID）")
print(f"トークンID: {token_ids}")

# ✅ ステップ3c: ID → トークン → テキストに復元
recovered_tokens = tokenizer.convert_ids_to_tokens(token_ids)
recovered_text = tokenizer.convert_tokens_to_string(recovered_tokens)
print(f"\nステップ3c: テキストに復元（ID → トークン → テキスト）")
print(f"復元テキスト: {recovered_text}")

# ✅ ステップ4: encode を使った一括処理
# 特殊トークン [CLS] と [SEP] を自動追加
encoded = tokenizer.encode(text, add_special_tokens=True)
print(f"\nステップ4: encode（特殊トークン付き）")
print(f"エンコード結果: {encoded}")

In [ ]:
# ✅ ステップ1: バッチ処理（複数のテキストを効率的に処理）
print("\n" + "="*60)
print("バッチ処理と パディング・截断")
print("="*60)

# ✅ ステップ2: 異なる長さのテキストを準備
texts_batch = [
    "Short text",
    "This is a medium length text that is a bit longer",
    "This is a much longer text " * 5  # 長いテキスト
]

# ✅ ステップ3: バッチ処理（すべてのテキストを同時に処理）
# padding=True: 短いテキストを長さに合わせて埋める
# truncation=True: 長いテキストを max_length に切り詰める
encoded_batch = tokenizer(
    texts_batch,
    padding=True,      # パディング
    truncation=True,   # 截断
    max_length=50,
    return_tensors='pt'
)

# ✅ ステップ4: バッチ処理の結果を表示
print(f"\n入力テキスト数: {len(texts_batch)}")
for i, text in enumerate(texts_batch):
    print(f"  テキスト{i+1}: {text[:40]}... (長さ: {len(text)}文字)")

## 3️⃣ 埋め込みベクトルの可視化

Embedding がテキストを数値で表現する方法を可視化

In [ ]:
# ✅ ステップ1: 埋め込みモデルをロード
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embeddings(texts):
    """✅ ステップ2: テキストの埋め込みベクトルを取得
    
    処理フロー:
    1. Tokenizer でテキストをトークン化
    2. モデルに入力して隠れ状態を取得
    3. [CLS] トークン（文全体の表現）を抽出
    """
    encoded = tokenizer(
        texts,
        padding=True,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    
    with torch.no_grad():
        output = model(**encoded)
    
    # [CLS] トークンの埋め込みを使用（文全体の表現）
    embeddings = output.last_hidden_state[:, 0, :]
    return embeddings.numpy()

# ✅ ステップ3: テスト用センテンスを準備
test_sentences = [
    "The cat is sleeping",
    "A dog is playing",
    "The weather is nice",
    "I like programming",
    "Python is great"
]

# ✅ ステップ4: すべてのセンテンスを埋め込み化
embeddings = get_embeddings(test_sentences)

print("="*60)
print("埋め込みベクトルの情報")
print("="*60)
print(f"\n埋め込み形状: {embeddings.shape}")
print(f"  テキスト数: {embeddings.shape[0]}")
print(f"  ベクトル次元: {embeddings.shape[1]}")

print(f"\n最初のセンテンスの埋め込み（最初の10次元）:")
print(f"  {embeddings[0, :10]}")

# ✅ ステップ5: センテンスペア間の類似性を計算
print(f"\n" + "="*60)
print("センテンスペアの類似性")
print("="*60)

similarity_matrix = cosine_similarity(embeddings)  # type: ignore

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"S{i+1}" for i in range(len(test_sentences))],
    columns=[f"S{i+1}" for i in range(len(test_sentences))]
)

print("\n類似性マトリックス（0～1、1に近いほど似ている）:")
print(similarity_df.round(3))

In [ ]:
# ✅ ステップ1: 高次元の埋め込みを 2D に次元削減して可視化
from sklearn.decomposition import PCA

# ✅ ステップ2: ランダムシードを固定（再現性を確保）
np.random.seed(42)
torch.manual_seed(42)

# ✅ ステップ3: PCA で高次元ベクトルを 2 次元に削減
# n_components=2: 2つの主要な軸を抽出
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(embeddings)

print(f"PCA 計算完了")
print(f"第1主成分の説明分散: {pca.explained_variance_ratio_[0]:.1%}")
print(f"第2主成分の説明分散: {pca.explained_variance_ratio_[1]:.1%}")
print(f"合計説明分散: {sum(pca.explained_variance_ratio_):.1%}\n")

# ✅ ステップ4: 散布図を描画
fig, ax = plt.subplots(figsize=(10, 7))

# ✅ ステップ5: 2D座標にプロット
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     s=200, alpha=0.7, c=range(len(test_sentences)), 
                     cmap='viridis', edgecolors='black', linewidth=1.5)

# ✅ ステップ6: 各ポイントにラベルを付ける
for i, sentence in enumerate(test_sentences):
    ax.annotate(f"S{i+1}: {sentence[:15]}", 
                xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', 
                facecolor='yellow', alpha=0.3))

# ✅ ステップ7: グラフのラベルとタイトルを設定
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=12)
ax.set_title("2D Visualization of Sentence Embeddings (PCA)", fontsize=14, fontweight='bold')

## 4️⃣ 簡易 RAG システムの実装

ベクトル検索を使った RAG システム

In [ ]:
class SimpleRAG:
    """✅ ステップ1: 簡易 RAG システムを実装"""
    
    def __init__(self):
        # ✅ ステップ2: 埋め込みモデルをロード
        self.tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
        self.model = AutoModel.from_pretrained('distilbert-base-uncased')
        
        # ✅ ステップ3: 知識ベース（検索対象のドキュメント）を定義
        self.knowledge_base = [
            "Machine learning is a subset of artificial intelligence.",
            "Deep learning uses neural networks with multiple layers.",
            "Natural language processing (NLP) deals with text analysis.",
            "Transformers are neural architectures based on attention mechanisms.",
            "TensorFlow and PyTorch are popular deep learning frameworks.",
        ]
        
        # ✅ ステップ4: 知識ベースを事前に埋め込み化して保存（高速化）
        self.knowledge_embeddings = self._embed_texts(self.knowledge_base)
    
    def _embed_texts(self, texts):
        """✅ ステップ5: テキストをベクトル化"""
        encoded = self.tokenizer(
            texts,
            padding=True,
            return_tensors='pt',
            truncation=True,
            max_length=512
        )
        
        with torch.no_grad():
            output = self.model(**encoded)
        
        return output.last_hidden_state[:, 0, :].numpy()
    
    def retrieve(self, query, top_k=2):
        """✅ ステップ6: クエリに関連したドキュメントを検索
        
        検索フロー:
        1. クエリをベクトル化
        2. 知識ベースのベクトルとの類似度を計算
        3. 類似度上位K件を返却
        """
        # クエリを埋め込み化
        query_embedding = self._embed_texts([query])[0]
        
        # コサイン類似度計算
        similarities: Any = cosine_similarity([query_embedding], self.knowledge_embeddings)[0]  # type: ignore
        
        # トップ-K を取得
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [
            {
                'text': self.knowledge_base[i],
                'score': similarities[i]
            }
            for i in top_indices
        ]

# ✅ ステップ7: RAG システムを初期化
rag = SimpleRAG()

print("="*60)
print("簡易 RAG システムのデモ")
print("="*60)

# ✅ ステップ8: テストクエリを実行
queries = [
    "What is machine learning?",
    "Tell me about deep learning",
    "Python programming"
]

for query in queries:
    print(f"\n質問: {query}")
    print("-" * 60)

## 5️⃣ 自律性スコアリング

AI の自立性を測定するスコアラー

In [ ]:
class AutonomyScorer:
    """✅ ステップ1: AI の自律性をスコアリングするクラス"""
    
    def __init__(self):
        # ✅ ステップ2: 自律性の評価基準を定義
        self.criteria = [
            'planning',        # 計画性：目標設定・計画能力
            'learning',        # 学習性：新知識の習得速度
            'adaptation',      # 適応性：環境変化への対応
            'decision_making', # 判断性：意思決定能力
            'explainability',  # 説明性：結果の説明可能性
            'efficiency',      # 効率性：リソース利用効率
            'reliability',     # 信頼性：安定性
            'robustness',      # 頑健性：エラー耐性
            'exploration'      # 探索性：新規領域の試行
        ]
        self.scores = {criterion: 0.0 for criterion in self.criteria}
    
    def rate_criterion(self, criterion_name, score):
        """✅ ステップ3: 各基準にスコアを設定（0～100）"""
        if criterion_name in self.scores:
            self.scores[criterion_name] = min(max(score, 0), 100)
    
    def calculate_overall_score(self):
        """✅ ステップ4: 全体スコアを計算"""
        return sum(self.scores.values()) / len(self.scores)
    
    def get_report(self):
        """✅ ステップ5: スコアリングレポートを作成"""
        overall = self.calculate_overall_score()
        
        report = {
            'overall_score': overall,
            'criterion_scores': self.scores.copy()
        }
        
        return report

# ✅ ステップ6: スコアラーを初期化
scorer = AutonomyScorer()

# ✅ ステップ7: サンプルスコアを設定
print("="*60)
print("自律性スコアリングの例")
print("="*60)

sample_scores = {
    'planning': 75,
    'learning': 82,
    'adaptation': 70,
    'decision_making': 88,
    'explainability': 79,
    'efficiency': 85,
    'reliability': 92,
    'robustness': 76,
    'exploration': 68
}

for criterion, score in sample_scores.items():
    scorer.rate_criterion(criterion, score)

# ✅ ステップ8: レポートを表示
report = scorer.get_report()

print(f"\n📊 自律性スコアリングレポート")
print(f"\n全体スコア: {report['overall_score']:.1f}/100")
print(f"\n基準別スコア:")
print("-" * 40)

df_scores = pd.DataFrame([

## 6️⃣ インタラクティブな演習問題

学んだ内容を実践してみましょう

In [ ]:
# ✅ ステップ1: 演習問題 1 - Tokenization
# 問題: 与えられたテキストをトークン化してください
print("\n" + "="*60)
print("演習問題 1: Tokenization")
print("="*60)
print("\n問題: 次のテキストをトークン化してください")
print("\nテキスト: 'I love learning about machine learning!'")

# ✅ ステップ2: 解答を実装
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
test_text = "I love learning about machine learning!"
tokens = tokenizer.tokenize(test_text)

print(f"\n解答:")
print(f"  トークン: {tokens}")
print(f"  トークン数: {len(tokens)}")

In [ ]:
# ✅ ステップ1: 演習問題 2 - 類似性の計算
print("\n" + "="*60)
print("演習問題 2: 類似性の計算")
print("="*60)
print("\n問題: 次の2つのセンテンスの類似性を計算してください")
print("\nセンテンス1: 'I like cats'")
print("センテンス2: 'I love dogs'")

# ✅ ステップ2: 埋め込みを取得して類似性を計算
sentences = ["I like cats", "I love dogs"]
embeds = get_embeddings(sentences)
sim = cosine_similarity(embeds)[0][1]

# ✅ ステップ3: 結果を表示
print(f"\n解答:")
print(f"  類似度: {sim:.4f}")
print(f"  解釈: {'高い類似性' if sim > 0.7 else '中程度の類似性' if sim > 0.5 else '低い類似性'}")

In [ ]:
# ✅ ステップ1: 演習問題 3 - RAG の実装
print("\n" + "="*60)
print("演習問題 3: RAG の実装")
print("="*60)
print("\n問題: 質問 'What are neural networks?' に対して")
print("知識ベースから最も関連性の高い情報を取得してください")

# ✅ ステップ2: RAG で知識ベースから情報を検索
query = "What are neural networks?"
results = rag.retrieve(query, top_k=1)

# ✅ ステップ3: 結果を表示
print(f"\n解答:")
print(f"  クエリ: {query}")
print(f"  取得した知識: {results[0]['text']}")
print(f"  関連性スコア: {results[0]['score']:.4f}")

## 🎓 学習のまとめ

このノートブックで学んだことをまとめます

## 🐛 Jupyterでのデバッグ方法

ブレイクポイント設定とデバッグテクニック


In [5]:
# ✅ ステップ1: Jupyterでのデバッグ方法

# 方法1: pdb.set_trace() を使用（プログラムが一時停止）
print("="*60)
print("デバッグ方法 1: pdb.set_trace()")
print("="*60)
print("""
使用方法:
  import pdb
  
  def debug_example(x):
      pdb.set_trace()  # ここで実行が一時停止
      result = x * 2
      return result
  
  debug_example(5)

コマンド:
  n  - 次の行へ
  c  - 継続実行
  p x - 変数 x を表示
  l  - コードリストを表示
  q  - デバッガを終了
""")

# ✅ ステップ2: 実例
import pdb

def example_function(x):
    # pdb.set_trace()  # コメント解除するとここで停止
    y = x + 10
    z = y * 2
    return z

result = example_function(5)
print(f"結果: {result}")

# ✅ ステップ3: IPython %debug マジックコマンド
print("\n" + "="*60)
print("デバッグ方法 2: %debug マジックコマンド")
print("="*60)
print("""
エラーが発生した直後に実行:
  %debug
  
これにより、エラー発生時のスタックトレースをデバッグできます。
""")

# ✅ ステップ4: try-except でエラー捕捉
print("\n" + "="*60)
print("デバッグ方法 3: try-except + デバッグプリント")
print("="*60)

def safe_division(a, b):
    try:
        print(f"入力: a={a}, b={b}")  # デバッグプリント
        result = a / b
        print(f"計算結果: {result}")  # デバッグプリント
        return result
    except ZeroDivisionError as e:
        print(f"❌ エラー発生: {e}")
        return None

safe_division(10, 2)
safe_division(10, 0)

# ✅ ステップ5: print デバッグ（最も簡単）
print("\n" + "="*60)
print("デバッグ方法 4: print デバッグ（推奨）")
print("="*60)

def analyze_text(text):
    print(f"📝 入力: {text}")
    
    tokens = text.split()
    print(f"📊 トークン数: {len(tokens)}")
    print(f"📋 トークン: {tokens}")
    
    upper_text = text.upper()
    print(f"🔤 大文字: {upper_text}")
    
    return len(tokens)

token_count = analyze_text("Hello World LLM")
print(f"✅ 最終結果: {token_count}")


デバッグ方法 1: pdb.set_trace()

使用方法:
  import pdb

  def debug_example(x):
      pdb.set_trace()  # ここで実行が一時停止
      result = x * 2
      return result

  debug_example(5)

コマンド:
  n  - 次の行へ
  c  - 継続実行
  p x - 変数 x を表示
  l  - コードリストを表示
  q  - デバッガを終了

結果: 30

デバッグ方法 2: %debug マジックコマンド

エラーが発生した直後に実行:
  %debug

これにより、エラー発生時のスタックトレースをデバッグできます。


デバッグ方法 3: try-except + デバッグプリント
入力: a=10, b=2
計算結果: 5.0
入力: a=10, b=0
❌ エラー発生: division by zero

デバッグ方法 4: print デバッグ（推奨）
📝 入力: Hello World LLM
📊 トークン数: 3
📋 トークン: ['Hello', 'World', 'LLM']
🔤 大文字: HELLO WORLD LLM
✅ 最終結果: 3


In [ ]:
# ✅ ステップ1: VS Code Jupyter デバッガの有効化
print("="*60)
print("VS Code Jupyter デバッガ設定")
print("="*60)

print("""
VS Code でのデバッグ設定:

1. 拡張機能をインストール
   - Jupyter（Microsoft 公式）
   - Python（Microsoft 公式）
   - Pylance

2. settings.json で設定
   "jupyter.debuggingEnabled": true

3. デバッグモードの有効化
   - コマンドパレット: Ctrl+Shift+P
   - "Jupyter: Enable Debugging" を実行

4. ブレイクポイント設定
   - 行番号の左側をクリック（赤い点が表示）
   - セル実行時に自動で停止
""")

# ✅ ステップ2: デバッグモードの確認
import sys
print(f"\nPython バージョン: {sys.version}")
print(f"デバッガ: pdb (標準ライブラリ)")

# ✅ ステップ3: デバッグ用の便利関数
def debug_print(label, value):
    """デバッグ出力用の便利関数"""
    print(f"🔍 [{label}] = {value} (型: {type(value).__name__})")

# テスト
debug_print("テスト変数", [1, 2, 3])
debug_print("テキスト", "Hello")
debug_print("数値", 42)

# ✅ ステップ4: デバッグ検査用の関数
def inspect_variable(var):
    """変数の詳細情報を表示"""
    print(f"\n📋 変数の詳細情報:")
    print(f"  値: {var}")
    print(f"  型: {type(var)}")
    print(f"  ID: {id(var)}")
    
    if hasattr(var, '__len__'):
        print(f"  長さ: {len(var)}")
    
    if isinstance(var, dict):
        print(f"  キー: {list(var.keys())}")
    
    if isinstance(var, (list, tuple)):
        print(f"  要素数: {len(var)}")
        print(f"  最初の要素: {var[0] if var else 'N/A'}")

# テスト
inspect_variable([1, 2, 3, 4, 5])


In [7]:
# ✅ ステップ1: 実践的なデバッグ例

# 前のセルで定義した関数を再定義（セル間の依存関係解消）
def debug_print(label, value):
    """デバッグ出力用の便利関数"""
    print(f"🔍 [{label}] = {value} (型: {type(value).__name__})")

def inspect_variable(var):
    """変数の詳細情報を表示"""
    print(f"\n📋 変数の詳細情報:")
    print(f"  値: {var}")
    print(f"  型: {type(var)}")
    print(f"  ID: {id(var)}")
    
    if hasattr(var, '__len__'):
        print(f"  長さ: {len(var)}")
    
    if isinstance(var, dict):
        print(f"  キー: {list(var.keys())}")
    
    if isinstance(var, (list, tuple)):
        print(f"  要素数: {len(var)}")
        print(f"  最初の要素: {var[0] if var else 'N/A'}")

print("="*60)
print("実践的なデバッグ例")
print("="*60)

# ✅ ステップ2: テキスト処理関数（バグ含む）
def process_texts(texts):
    """テキスト処理関数"""
    results = []
    
    for i, text in enumerate(texts):
        debug_print(f"処理中 [{i}]", text)
        
        # テキストの処理
        cleaned = text.strip().lower()
        words = cleaned.split()
        
        debug_print(f"単語数", len(words))
        
        result = {
            'original': text,
            'cleaned': cleaned,
            'word_count': len(words),
            'words': words
        }
        results.append(result)
    
    return results

# ✅ ステップ3: デバッグ実行
test_texts = [
    "  Hello World  ",
    "Python Programming",
    "Machine Learning AI"
]

processed = process_texts(test_texts)

# ✅ ステップ4: 結果を検査
print("\n結果:")
for i, item in enumerate(processed):
    inspect_variable(item)

# ✅ ステップ5: ブレイクポイント例
print("\n" + "="*60)
print("ブレイクポイント設定場所の例")
print("="*60)

def example_with_breakpoint(x):
    """ここでブレイクポイントをセット"""
    # 👇 この行の左側をクリックして赤点を付ける
    y = x * 2                    # ← ブレイクポイント位置
    z = y + 10
    print(f"x={x}, y={y}, z={z}")
    return z

# コメント解除してセル実行 → ブレイクポイントで停止
# result = example_with_breakpoint(5)

print("ℹ️  ブレイクポイント設定方法:")
print("  1. 行番号の左側をクリック")
print("  2. 赤い円が表示される")
print("  3. セルを実行すると、その行で停止")
print("  4. ステップ実行 (Step Over) で1行ずつ実行可能")

実践的なデバッグ例
🔍 [処理中 [0]] =   Hello World   (型: str)
🔍 [単語数] = 2 (型: int)
🔍 [処理中 [1]] = Python Programming (型: str)
🔍 [単語数] = 2 (型: int)
🔍 [処理中 [2]] = Machine Learning AI (型: str)
🔍 [単語数] = 3 (型: int)

結果:

📋 変数の詳細情報:
  値: {'original': '  Hello World  ', 'cleaned': 'hello world', 'word_count': 2, 'words': ['hello', 'world']}
  型: <class 'dict'>
  ID: 125515588565504
  長さ: 4
  キー: ['original', 'cleaned', 'word_count', 'words']

📋 変数の詳細情報:
  値: {'original': 'Python Programming', 'cleaned': 'python programming', 'word_count': 2, 'words': ['python', 'programming']}
  型: <class 'dict'>
  ID: 125515588739648
  長さ: 4
  キー: ['original', 'cleaned', 'word_count', 'words']

📋 変数の詳細情報:
  値: {'original': 'Machine Learning AI', 'cleaned': 'machine learning ai', 'word_count': 3, 'words': ['machine', 'learning', 'ai']}
  型: <class 'dict'>
  ID: 125515589228032
  長さ: 4
  キー: ['original', 'cleaned', 'word_count', 'words']

ブレイクポイント設定場所の例
ℹ️  ブレイクポイント設定方法:
  1. 行番号の左側をクリック
  2. 赤い円が表示される
  3. セルを実行すると、その行で停止

In [ ]:
summary = """
🎓 このノートブックで学んだこと
================================

✅ 1. 最初の推論 (Hello LLM)
   - パイプラインを使った簡単な推論実装
   - テキスト分類、質問応答など複数のタスク

✅ 2. Tokenizer の詳細
   - トークン化のプロセス
   - 特殊トークン ([CLS], [SEP])
   - バッチ処理でのパディング・截断

✅ 3. 埋め込みベクトル
   - テキストを数値ベクトルに変換
   - コサイン類似度で意味的な近さを計算
   - 高次元ベクトルの 2D 可視化

✅ 4. RAG システム
   - ベクトル検索を使った関連知識の抽出
   - 知識ベースからの効率的な情報取得

✅ 5. 自律性スコアリング
   - 9つの基準での AI 評価
   - 全体スコアの計算

📚 次のステップ
==============
- プロジェクトの実装を読む
- カスタム知識ベースを追加
- モデルをファインチューニング
- 独自の評価指標を実装
"""

print(summary)

---

## 🚀 学習完了おめでとうございます！

このノートブックを通じて、LLM の基本から応用まで実装レベルで学習しました。

### 次に進むべきことは:
1. **プロジェクトのコード読解** - autonomous_rag_agent.py などの実装を学ぶ
2. **カスタマイズ** - 自分のデータで RAG を実装
3. **拡張** - 新しい機能を追加
4. **デプロイ** - アプリケーションをビルド

頑張ってください！
   